# 🔍 Depth Estimation Pipeline
**YOLOv8n + Depth Anything V2 + DINOv2**

Works on:
- ✅ Google Colab (free CPU or T4 GPU)
- ✅ Mac M1/M2/M3 (uses MPS automatically)
- ✅ Any Linux/Windows machine

---
### Before you start (Colab only)
Go to **Runtime → Change runtime type** and pick:
- **T4 GPU** — fastest, recommended
- **CPU** — works but slow (~5-10 min/image)

In [1]:
# ── Cell 1: Detect environment ────────────────────────────────────────────────
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
try:
    import torch
    if torch.cuda.is_available():
        DEVICE = 'cuda'
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        DEVICE = 'mps'
    else:
        DEVICE = 'cpu'
except ImportError:
    DEVICE = 'cpu'

print(f'Environment : {"Google Colab" if IN_COLAB else "Local"}')
print(f'Device      : {DEVICE}')

Environment : Google Colab
Device      : cuda


In [2]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
# Takes ~2-3 min on first run. Safe to re-run.

# PyTorch — skip if already correct version
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('torch>=2.1.0', 'torchvision>=0.16.0')
pip('transformers>=4.40.0', 'huggingface_hub>=0.22.0')
pip('ultralytics>=8.2.0')
pip('opencv-python-headless>=4.9.0')  # headless = no display needed (Colab/server safe)
pip('Pillow>=10.0.0', 'numpy>=1.24.0', 'matplotlib>=3.8.0', 'tqdm')

print('\n✅ All packages installed')


✅ All packages installed


In [4]:
# ── Cell 3: Upload depth_pipeline.py ─────────────────────────────────────────
# Colab sometimes renames files to 'depth_pipeline (2).py' etc — this handles it.
# Local users: skip this cell if depth_pipeline.py is already in the same folder.

import sys, shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import files
    print('Upload depth_pipeline.py when the dialog appears …')
    uploaded = files.upload()

    # Colab renames duplicates: 'depth_pipeline (3).py' etc — match any variant
    py_files = [f for f in uploaded if 'depth_pipeline' in f and f.endswith('.py')]
    if not py_files:
        raise FileNotFoundError('No depth_pipeline*.py found in upload. Please upload depth_pipeline.py')

    uploaded_name = py_files[0]
    if uploaded_name != 'depth_pipeline.py':
        shutil.copy(uploaded_name, 'depth_pipeline.py')
        print(f'  (Colab renamed it to {uploaded_name!r} — copied to depth_pipeline.py)')

    print('✅ depth_pipeline.py ready')
else:
    if not Path('depth_pipeline.py').exists():
        raise FileNotFoundError('depth_pipeline.py not found. Put it in the same folder as this notebook.')
    print('✅ depth_pipeline.py found locally')


Upload depth_pipeline.py when the dialog appears …


Saving depth_pipeline.py to depth_pipeline.py
✅ depth_pipeline.py ready


In [5]:
# ── Cell 4: Upload your dataset images ───────────────────────────────────────
# Creates a 'dataset/' folder and puts your images in it.
# OPTION A — Colab file upload (one or more images)
# OPTION B — Local: your 'dataset/' folder already exists, skip this cell

import os, shutil
from pathlib import Path

DATASET_DIR = Path('dataset')
DATASET_DIR.mkdir(exist_ok=True)

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import files
    print('Select your images (jpg/png/webp etc) — you can pick multiple files:')
    uploaded = files.upload()

    img_exts = {'.jpg','.jpeg','.png','.bmp','.webp','.tiff'}
    moved = []
    for fname in uploaded:
        if Path(fname).suffix.lower() in img_exts:
            dest = DATASET_DIR / fname
            shutil.move(fname, dest)
            moved.append(dest.name)

    print(f'\n✅ {len(moved)} image(s) moved to dataset/:')
    for m in moved: print(f'   {m}')
else:
    imgs = list(DATASET_DIR.glob('*'))
    img_exts = {'.jpg','.jpeg','.png','.bmp','.webp','.tiff'}
    imgs = [f for f in imgs if f.suffix.lower() in img_exts]
    if not imgs:
        print('⚠️  dataset/ is empty — add images to the dataset/ folder then re-run')
    else:
        print(f'✅ Found {len(imgs)} image(s) in dataset/:')
        for f in imgs: print(f'   {f.name}')

Select your images (jpg/png/webp etc) — you can pick multiple files:


Saving IMG_0557.JPG to IMG_0557.JPG
Saving IMG_0558.jpg to IMG_0558.jpg
Saving IMG_0559.JPG to IMG_0559.JPG
Saving IMG_0560.jpg to IMG_0560.jpg
Saving IMG_6324.heic to IMG_6324.heic
Saving IMG_6325.heic to IMG_6325.heic
Saving IMG_6326.heic to IMG_6326.heic
Saving IMG_6327.heic to IMG_6327.heic

✅ 4 image(s) moved to dataset/:
   IMG_0557.JPG
   IMG_0558.jpg
   IMG_0559.JPG
   IMG_0560.jpg


In [6]:
# ── Cell 5: Run the pipeline ──────────────────────────────────────────────────
# Models are downloaded from HuggingFace on first run (~500 MB total).
# Cached automatically — subsequent runs are instant.
#
# Model size guide:
#   Small  → fastest, less accurate  (default, good for Colab CPU)
#   Base   → balanced
#   Large  → best quality, needs GPU

import sys
sys.argv = [
    'depth_pipeline.py',
    '--input',   'dataset',
    '--output',  'output',
    '--depth',   'depth-anything/Depth-Anything-V2-Small-hf',  # change to Base/Large if you have GPU
    '--dino',    'facebook/dinov2-base',
    '--colormap','inferno',
    # '--device', 'mps',  # ← uncomment on Mac M1/M2/M3
]

exec(open('depth_pipeline.py').read())


════════════════════════════════════════════════════════════
  Depth Estimation Pipeline
  Device  : cuda
  YOLO    : yolov8n.pt
  Depth   : depth-anything/Depth-Anything-V2-Small-hf
  DINOv2  : facebook/dinov2-base
════════════════════════════════════════════════════════════

[Loading models]
  ▶ YOLOv8n   loading 'yolov8n.pt' …
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
  ▶ DepthAny  loading 'depth-anything/Depth-Anything-V2-Small-hf' …


preprocessor_config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/99.2M [00:00<?, ?B/s]

...

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]


[Models ready]

Found 4 image(s)  →  output: 'output'

────────────────────────────────────────────────────────────
  IMG_0557.JPG
  [1/3] Depth Anything V2 … done  (min=0.00, max=12.47)
  [2/3] YOLOv8n detection … done  (11 object(s) detected)
  [3/3] DINOv2 features … done
  Detections:
  [car            ]  conf=0.74  bbox=(1445,2734,1982,2939)  depth_median=0.0594  depth_std=0.0055  dino‖feat‖=48.20
  [car            ]  conf=0.70  bbox=(3011,2732,3245,2925)  depth_median=0.0607  depth_std=0.0043  dino‖feat‖=48.29
  [car            ]  conf=0.69  bbox=(2075,2724,2293,2913)  depth_median=0.0554  depth_std=0.0047  dino‖feat‖=47.27
  [car            ]  conf=0.63  bbox=(2398,2744,2661,2927)  depth_median=0.0564  depth_std=0.0041  dino‖feat‖=47.57
  [car            ]  conf=0.60  bbox=(3640,2742,3975,2927)  depth_median=0.0652  depth_std=0.0050  dino‖feat‖=47.18
  [car            ]  conf=0.53  bbox=(846,2739,1139,2928)  depth_median=0.0463  depth_std=0.0032  dino‖feat‖=44.14
  [car        

In [7]:
# ── Cell 6: Preview results inline ───────────────────────────────────────────
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

composites = sorted(Path('output').glob('*_composite.jpg'))

if not composites:
    print('No composites found — did the pipeline run successfully?')
else:
    for comp in composites:
        img = mpimg.imread(str(comp))
        fig, ax = plt.subplots(figsize=(18, 5))
        ax.imshow(img)
        ax.set_title(comp.stem, fontsize=12, pad=8)
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        print()

In [8]:
# ── Cell 7: Show JSON results ─────────────────────────────────────────────────
import json
from pathlib import Path

json_path = Path('output/results.json')
if json_path.exists():
    data = json.loads(json_path.read_text())
    for item in data:
        print(f"\n📷 {item['file']}  ({item['image_size'][0]}x{item['image_size'][1]})")
        print(f"   Detections: {item['n_detections']}")
        for d in item['detections']:
            print(f"   • {d['class']:<15s}  conf={d['confidence']:.2f}  "
                  f"depth_median={d['depth_median']:.4f}  "
                  f"dino_norm={d['dino_feat_norm']:.2f}")
else:
    print('results.json not found yet — run Cell 5 first.')


📷 IMG_0557.JPG  (4110x5480)
   Detections: 11
   • car              conf=0.74  depth_median=0.0594  dino_norm=48.20
   • car              conf=0.70  depth_median=0.0607  dino_norm=48.29
   • car              conf=0.69  depth_median=0.0554  dino_norm=47.27
   • car              conf=0.63  depth_median=0.0564  dino_norm=47.57
   • car              conf=0.60  depth_median=0.0652  dino_norm=47.18
   • car              conf=0.53  depth_median=0.0463  dino_norm=44.14
   • car              conf=0.52  depth_median=0.0518  dino_norm=45.19
   • car              conf=0.49  depth_median=0.0515  dino_norm=42.40
   • car              conf=0.42  depth_median=0.0531  dino_norm=44.81
   • car              conf=0.34  depth_median=0.0651  dino_norm=43.57
   • car              conf=0.32  depth_median=0.0543  dino_norm=41.45

📷 IMG_0558.jpg  (4052x5403)
   Detections: 10
   • car              conf=0.72  depth_median=0.0711  dino_norm=47.95
   • car              conf=0.69  depth_median=0.0732  dino_norm=48

In [9]:
# ── Cell 8: Download all outputs (Colab only) ─────────────────────────────────
# Zips the output/ folder and downloads it to your computer.
# Skip this cell if running locally — files are already in output/

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import shutil
    shutil.make_archive('depth_pipeline_output', 'zip', 'output')
    from google.colab import files
    files.download('depth_pipeline_output.zip')
    print('✅ Download started')
else:
    print('Local run — find your results in the output/ folder.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started
